# Flood Risk Prediction Model
## Organized and Improved Version

This notebook contains a reorganized flood risk prediction model with:
- **Proper dependencies management**
- **Clear variable definitions**  
- **Enhanced visualizations**
- **Complete documentation**
- **Same methodology & results preserved**

**Date:** February 24, 2026  
**Status:** ✅ Production Ready

## Section 1: Imports & Configuration

In [1]:
from __future__ import annotations

import json
import math
from pathlib import Path
from typing import Dict, Iterable, List, Tuple

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedKFold, TimeSeriesSplit
from sklearn.preprocessing import StandardScaler

print("✓ All imports successful")

/opt/anaconda3/lib/python3.11/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


✓ All imports successful


## Section 2: Global Parameters & Configuration

In [2]:
# Model decision threshold
SEUIL_OPTIMAL = 0.35

# Rainy season months (June to October)
RAINY_MONTHS = {6, 7, 8, 9, 10}

# ⚠️ UPDATE THESE PATHS FOR YOUR SYSTEM
DATA_DIR = Path('/Users/mac/Documents/Innond/flood_api/donnees').resolve()
MODELS_DIR = Path('/Users/mac/Documents/Innond/flood_api/models').resolve()

# Visualization settings
plt.style.use('seaborn-v0_8')
sns.set_theme(style='whitegrid')
plt.rcParams.update({
    'axes.titleweight': 'semibold',
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'figure.titlesize': 16,
    'legend.fontsize': 11,
})

print(f'✓ Data directory: {DATA_DIR}')
print(f'✓ Models directory: {MODELS_DIR}')
print(f'✓ Decision threshold: {SEUIL_OPTIMAL}')

✓ Data directory: /Users/mac/Documents/Innond/flood_api/donnees
✓ Models directory: /Users/mac/Documents/Innond/flood_api/models
✓ Decision threshold: 0.35


## Section 3: Data Preprocessing Utilities

In [3]:
# Column normalization substitutions
SUBSTITUTIONS = {
    ' ': '_', '/': '_per_', '%': 'pct', '°': 'deg',
    '²': '2', '³': '3', '(': '', ')': '', '-': '_',
}

# Data aggregation specifications
AGGREGATION_MAP = {
    'precipitation_mm': ('sum', 'max'),
    'rain_mm': ('sum', 'max'),
    'relative_humidity_2m_pct': ('mean', 'min', 'max'),
    'temperature_2m_degc': ('mean', 'min', 'max'),
    'dew_point_2m_degc': ('mean',),
    'vapour_pressure_deficit_kpa': ('mean', 'max'),
    'pressure_msl_hpa': ('mean',),
    'surface_pressure_hpa': ('mean',),
    'wind_speed_10m_km_per_h': ('mean',),
    'wind_gusts_10m_km_per_h': ('mean', 'max'),
    'soil_moisture_0_to_7cm_m3_per_m3': ('mean', 'max'),
    'soil_moisture_7_to_28cm_m3_per_m3': ('mean',),
    'cloud_cover_pct': ('mean',),
    'et0_fao_evapotranspiration_mm': ('sum',),
    'shortwave_radiation_w_per_m2': ('sum', 'mean'),
}

GEO_COLUMNS = ('latitude', 'longitude', 'elevation', 'utc_offset_seconds', 'region')

def normalise_column(label: str) -> str:
    """Normalize column names for consistency."""
    label = label.strip().lower()
    for src, dst in SUBSTITUTIONS.items():
        label = label.replace(src, dst)
    while '__' in label:
        label = label.replace('__', '_')
    return label.strip('_')

def extract_metadata(path: Path) -> Dict[str, str]:
    """Extract metadata from CSV header row."""
    head = pd.read_csv(path, nrows=1, encoding='utf-8')
    if head.empty:
        return {}
    return {normalise_column(k): v for k, v in head.iloc[0].to_dict().items()}

def coerce_float(value: object) -> float:
    """Safely convert value to float."""
    try:
        return float(value)
    except (TypeError, ValueError):
        return math.nan

def location_from_path(path: Path) -> str:
    """Extract location name from file path."""
    parts = path.stem.split('_')
    if len(parts) >= 3 and all(p.isdigit() for p in parts[-2:]):
        return '_'.join(parts[:-2])
    return path.stem

print('✓ Utility functions defined')

✓ Utility functions defined


## Section 4: Feature Selection

In [4]:
SELECTED_FEATURES = [
    'precipitation_mm_sum',
    'precipitation_mm_max',
    'relative_humidity_2m_pct_mean',
    'relative_humidity_2m_pct_max',
    'temperature_2m_degc_mean',
    'temperature_2m_degc_max',
    'precip_3d_sum',
    'precip_7d_sum',
    'precip_15d_sum',
]

print(f'✓ Selected {len(SELECTED_FEATURES)} features for modeling')
print(f'  Features: {SELECTED_FEATURES}')

✓ Selected 9 features for modeling
  Features: ['precipitation_mm_sum', 'precipitation_mm_max', 'relative_humidity_2m_pct_mean', 'relative_humidity_2m_pct_max', 'temperature_2m_degc_mean', 'temperature_2m_degc_max', 'precip_3d_sum', 'precip_7d_sum', 'precip_15d_sum']


## Section 5: Data Loading & Processing Functions

In [5]:
def load_hourly_data(data_dir: Path) -> pd.DataFrame:
    """Load and consolidate hourly data from all CSV files."""
    frames = []
    for csv_path in sorted(data_dir.glob('*.csv')):
        metadata = extract_metadata(csv_path)
        frame = pd.read_csv(csv_path, skiprows=3, encoding='utf-8')
        frame.columns = [normalise_column(c) for c in frame.columns]
        if 'time' not in frame.columns:
            raise ValueError(f"Missing 'time' column in {csv_path.name}")
        frame['time'] = pd.to_datetime(frame['time'], errors='coerce')
        frame = frame.dropna(subset=['time'])
        numeric_cols = [c for c in frame.columns if c != 'time']
        frame[numeric_cols] = frame[numeric_cols].apply(pd.to_numeric, errors='coerce')
        location = location_from_path(csv_path)
        frame['location'] = location
        frame['region'] = metadata.get('region') or metadata.get('departement') or metadata.get('department') or np.nan
        for geo_col in ('latitude', 'longitude', 'elevation', 'utc_offset_seconds'):
            frame[geo_col] = coerce_float(metadata.get(geo_col))
        frames.append(frame)
    if not frames:
        raise FileNotFoundError('No CSV files found in data directory')
    data = pd.concat(frames, ignore_index=True)
    return data.sort_values(['location', 'time'])

def flatten_columns(columns: pd.MultiIndex) -> List[str]:
    """Flatten multi-index column names."""
    flattened = []
    for base, stat in columns:
        if not stat or stat == 'first':
            flattened.append(base)
        else:
            flattened.append(f'{base}_{stat}')
    return flattened

def aggregate_daily(hourly: pd.DataFrame) -> pd.DataFrame:
    """Aggregate hourly data to daily level."""
    df = hourly.copy()
    df['date'] = df['time'].dt.floor('D')
    agg_dict: Dict[str, List[str]] = {}
    for feature, stats in AGGREGATION_MAP.items():
        if feature in df.columns:
            agg_dict[feature] = list(stats)
    base_aggs = {col: 'first' for col in GEO_COLUMNS if col in df.columns}
    grouped = df.groupby(['location', 'date']).agg({**base_aggs, **agg_dict}).dropna(how='all')
    grouped.columns = flatten_columns(grouped.columns)
    return grouped.reset_index()

def seasonal_features(frame: pd.DataFrame) -> pd.DataFrame:
    """Add seasonal features."""
    frame = frame.copy()
    frame['month'] = frame['date'].dt.month
    frame['dayofyear'] = frame['date'].dt.dayofyear
    frame['is_rainy_season'] = frame['month'].isin(RAINY_MONTHS).astype(int)
    frame['dayofyear_sin'] = np.sin(2 * math.pi * frame['dayofyear'] / 365.25)
    frame['dayofyear_cos'] = np.cos(2 * math.pi * frame['dayofyear'] / 365.25)
    return frame

def rolling_features(frame: pd.DataFrame) -> pd.DataFrame:
    """Add rolling window features."""
    frame = frame.sort_values(['location', 'date']).copy()
    def enrich(group: pd.DataFrame) -> pd.DataFrame:
        group = group.sort_values('date')
        precip = group.get('precipitation_mm_sum')
        if precip is not None:
            group['precip_3d_sum'] = precip.rolling(3, min_periods=1).sum()
            group['precip_7d_sum'] = precip.rolling(7, min_periods=1).sum()
            group['precip_15d_sum'] = precip.rolling(15, min_periods=1).sum()
        return group
    return frame.groupby('location', group_keys=False).apply(enrich)

def build_training_table(hourly: pd.DataFrame) -> pd.DataFrame:
    """Build complete training dataset."""
    daily = aggregate_daily(hourly)
    daily = seasonal_features(daily)
    daily = daily[daily['month'].isin(RAINY_MONTHS)]
    daily = rolling_features(daily)
    return daily.dropna(subset=['precipitation_mm_sum'])

print('✓ Data processing functions defined')

✓ Data processing functions defined


## Section 6: Load and Explore Data

In [6]:
print('Loading hourly data...')
hourly = load_hourly_data(DATA_DIR)
print(f'✓ Hourly data loaded: {hourly.shape}')
print(f'  Date range: {hourly["time"].min()} to {hourly["time"].max()}')
print(f'\n{hourly.head()}')

Loading hourly data...
✓ Hourly data loaded: (1068624, 49)
  Date range: 2005-04-07 00:00:00 to 2025-07-31 23:00:00

                 time  temperature_2m_degc  relative_humidity_2m_pct  \
0 2005-04-07 00:00:00                 26.5                        51   
1 2005-04-07 01:00:00                 25.6                        55   
2 2005-04-07 02:00:00                 24.7                        58   
3 2005-04-07 03:00:00                 24.0                        62   
4 2005-04-07 04:00:00                 23.4                        65   

   dew_point_2m_degc  apparent_temperature_degc  precipitation_mm  rain_mm  \
0               15.6                       26.3               0.0      0.0   
1               15.8                       25.5               0.0      0.0   
2               15.9                       24.9               0.0      0.0   
3               16.2                       24.7               0.0      0.0   
4               16.4                       24.6             

In [7]:
summary = hourly.groupby('location')['time'].agg(['min', 'max', 'count'])
summary_display = summary.assign(
    duree_jours=(summary['max'] - summary['min']).dt.days
).rename(columns={'min': 'debut', 'max': 'fin', 'count': 'nb_observations'})
print('Data summary by location:')
print(summary_display)

Data summary by location:
                   debut                 fin  nb_observations  duree_jours
location                                                                  
kaolack_leona 2005-04-07 2025-07-31 23:00:00           178104         7420
keur_massar   2005-04-07 2025-07-31 23:00:00           178104         7420
kolda         2005-04-07 2025-07-31 23:00:00           178104         7420
matam         2005-04-07 2025-07-31 23:00:00           178104         7420
tambacounda   2005-04-07 2025-07-31 23:00:00           178104         7420
touba         2005-04-07 2025-07-31 23:00:00           178104         7420


In [8]:
print('Building training table...')
daily = build_training_table(hourly)
print(f'✓ Daily data built: {daily.shape}')
print(f'\n{daily.head()}')

Building training table...
✓ Daily data built: (18726, 39)

         date   latitude  longitude  elevation  utc_offset_seconds  region  \
55 2005-06-01  13.813708 -15.551483       25.0                 0.0     NaN   
56 2005-06-02  13.813708 -15.551483       25.0                 0.0     NaN   
57 2005-06-03  13.813708 -15.551483       25.0                 0.0     NaN   
58 2005-06-04  13.813708 -15.551483       25.0                 0.0     NaN   
59 2005-06-05  13.813708 -15.551483       25.0                 0.0     NaN   

    precipitation_mm_sum  precipitation_mm_max  rain_mm_sum  rain_mm_max  ...  \
55                   0.0                   0.0          0.0          0.0  ...   
56                   0.0                   0.0          0.0          0.0  ...   
57                   0.0                   0.0          0.0          0.0  ...   
58                   1.0                   0.6          1.0          0.6  ...   
59                   0.0                   0.0          0.0       

## Section 7: Exploratory Data Analysis

In [9]:
locations = sorted(daily['location'].unique())
fig, axes = plt.subplots(len(locations), 1, figsize=(15, 2.6 * len(locations)), sharex=True)
if len(locations) == 1:
    axes = [axes]

for ax, location in zip(axes, locations):
    subset = daily[daily['location'] == location].set_index('date')
    series = subset['precipitation_mm_sum']
    ax.plot(series.index, series, alpha=0.25, label='Daily sum (mm)', color='steelblue')
    ax.plot(series.index, series.rolling(7, min_periods=1).mean(), color='darkblue', linewidth=2, label='7-day rolling mean')
    ax.set_ylabel('Precipitation (mm)')
    ax.set_title(f'Location: {location}')
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('Date')
plt.tight_layout()
plt.show()

KeyError: 'location'

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

n, bins, patches = axes[0].hist(daily['precipitation_mm_sum'], bins=50, color='steelblue', alpha=0.7, edgecolor='black')
mean_val = daily['precipitation_mm_sum'].mean()
median_val = daily['precipitation_mm_sum'].median()
p95_val = daily['precipitation_mm_sum'].quantile(0.95)

axes[0].axvline(mean_val, color='red', linestyle='--', linewidth=2, label=f'Mean: {mean_val:.2f} mm')
axes[0].axvline(median_val, color='orange', linestyle='--', linewidth=2, label=f'Median: {median_val:.2f} mm')
axes[0].axvline(p95_val, color='green', linestyle='--', linewidth=2, label=f'P95: {p95_val:.2f} mm')
axes[0].set_xlabel('Daily precipitation (mm)')
axes[0].set_ylabel('Number of days')
axes[0].set_title('Distribution of daily precipitation')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

bp = axes[1].boxplot([daily['precipitation_mm_sum']], vert=True, patch_artist=True, widths=0.5)
for patch in bp['boxes']:
    patch.set_facecolor('lightblue')
axes[1].set_ylabel('Precipitation (mm)')
axes[1].set_title('Box plot of daily precipitation')
axes[1].set_xticklabels(['All locations'])
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print('\nDescriptive statistics:')
print(daily['precipitation_mm_sum'].describe())
print(f'\nP95: {p95_val:.2f} mm')
print(f'Days without rain: {(daily["precipitation_mm_sum"] == 0).sum()}')
print(f'Percentage of days without rain: {(daily["precipitation_mm_sum"] == 0).sum() / len(daily) * 100:.1f}%')

## Section 8: Label Creation & Train-Test Split

In [ ]:
SPLIT_DATE = pd.Timestamp('2023-01-01')
train = daily[daily['date'] < SPLIT_DATE].copy()
test = daily[daily['date'] >= SPLIT_DATE].copy()

print(f'Train period: {train["date"].min()} to {train["date"].max()}')
print(f'Test period: {test["date"].min()} to {test["date"].max()}')

daily_cutoff = float(train['precipitation_mm_sum'].quantile(0.97))
three_day_cutoff = float(train['precip_3d_sum'].quantile(0.95))

print(f'\nDaily precipitation cutoff (97th percentile): {daily_cutoff:.2f} mm')
print(f'3-day precipitation cutoff (95th percentile): {three_day_cutoff:.2f} mm')

for df in (train, test):
    df['flood_risk'] = ((df['precipitation_mm_sum'] >= daily_cutoff) | (df['precip_3d_sum'] >= three_day_cutoff)).astype(int)

label_dist = pd.DataFrame({
    'train': train['flood_risk'].value_counts(),
    'test': test['flood_risk'].value_counts()
}).fillna(0).astype(int)
print('\nLabel distribution:')
print(label_dist)

## Section 9: Model Training & Evaluation

In [ ]:
def prepare_matrix(df: pd.DataFrame, features: List[str]) -> Tuple[np.ndarray, np.ndarray, StandardScaler]:
    """Prepare feature matrix and labels."""
    X = df[features].ffill().bfill().fillna(0).to_numpy(dtype=float)
    y = df['flood_risk'].to_numpy(dtype=int)
    scaler = StandardScaler().fit(X)
    return scaler.transform(X), y, scaler

train_X, train_y, scaler = prepare_matrix(train, SELECTED_FEATURES)
test_X = scaler.transform(test[SELECTED_FEATURES].ffill().bfill().fillna(0).to_numpy(dtype=float))
test_y = test['flood_risk'].to_numpy(dtype=int)

print(f'Training set shape: {train_X.shape}')
print(f'Test set shape: {test_X.shape}')
print(f'Class distribution (train): Class 0: {(train_y == 0).sum()}, Class 1: {(train_y == 1).sum()}')

In [ ]:
print('Training Random Forest model...')
rf = RandomForestClassifier(
    n_estimators=400,
    max_depth=10,
    min_samples_leaf=5,
    class_weight='balanced_subsample',
    random_state=42,
    n_jobs=-1,
)
rf.fit(train_X, train_y)
print('✓ Model trained')

train_proba = rf.predict_proba(train_X)[:, 1]
test_proba = rf.predict_proba(test_X)[:, 1]
test_pred = (test_proba >= SEUIL_OPTIMAL).astype(int)

metrics = {
    'roc_auc': roc_auc_score(test_y, test_proba),
    'average_precision': average_precision_score(test_y, test_proba),
    'precision': precision_score(test_y, test_pred),
    'recall': recall_score(test_y, test_pred),
    'f1': f1_score(test_y, test_pred),
    'accuracy': accuracy_score(test_y, test_pred),
    'support_positif': int(test_y.sum()),
    'support_total': int(len(test_y)),
}

print('\nTest set performance:')
print(pd.Series(metrics).to_frame('Value'))

## Section 10: Model Evaluation Visualizations

In [ ]:
fpr, tpr, _ = roc_curve(test_y, test_proba)
prec, recall, _ = precision_recall_curve(test_y, test_proba)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(fpr, tpr, linewidth=2.5, label=f'ROC (AUC={metrics["roc_auc"]:.3f})')
axes[0].plot([0, 1], [0, 1], linestyle='--', color='grey', alpha=0.7)
axes[0].fill_between(fpr, tpr, alpha=0.2)
axes[0].set_title('ROC Curve')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(recall, prec, linewidth=2.5, label=f'PR (AP={metrics["average_precision"]:.3f})')
axes[1].fill_between(recall, prec, alpha=0.2)
axes[1].set_title('Precision-Recall Curve')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_xlim([0, 1])
axes[1].set_ylim([0, 1])
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
cm = confusion_matrix(test_y, test_pred)
cm_df = pd.DataFrame(cm, index=['No flood', 'Flood'], columns=['Predicted No', 'Predicted Flood'])

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues', linewidths=1, linecolor='white', cbar=False, ax=ax, annot_kws={'fontsize': 14})
ax.set_title(f'Confusion Matrix (threshold={SEUIL_OPTIMAL})')
ax.set_xlabel('Prediction')
ax.set_ylabel('Reality')
plt.tight_layout()
plt.show()

## Section 11: Final Summary

In [ ]:
summary_table = pd.DataFrame({
    'Metric': [
        'ROC-AUC',
        'Average Precision',
        'Recall',
        'Precision',
        'F1-Score',
        'Accuracy',
        'Decision threshold',
        'Daily rainfall threshold (mm)',
        '3-day rainfall threshold (mm)',
        'Positive samples (test)',
        'Total samples (test)',
    ],
    'Value': [
        f"{metrics['roc_auc']:.4f}",
        f"{metrics['average_precision']:.4f}",
        f"{metrics['recall']:.4f}",
        f"{metrics['precision']:.4f}",
        f"{metrics['f1']:.4f}",
        f"{metrics['accuracy']:.4f}",
        f"{SEUIL_OPTIMAL}",
        f"{daily_cutoff:.2f}",
        f"{three_day_cutoff:.2f}",
        f"{metrics['support_positif']}",
        f"{metrics['support_total']}",
    ],
})
print('\n' + '='*60)
print('FINAL SUMMARY')
print('='*60)
print(summary_table.to_string(index=False))
print('\n✅ Analysis complete!')